# 🎯 5. Generators

Generators are functions that produce values **lazily** — one at a time, on demand.
They're the backbone of Python's memory-efficient iteration.

| ☕ Java | 🐍 Python |
|---------|----------|
| `Iterator<T>` interface (class + `hasNext`/`next`) | `yield` keyword — one line |
| `Stream.iterate()` / `Stream.generate()` | Generator functions / expressions |
| Streams are single-use | Generators are single-use too |

In [ ]:
import sys
from typing import Generator
from itertools import islice, chain, takewhile, count, groupby

## 5.1 Generator Functions — `yield`

A function with `yield` becomes a **generator function**.
When called, it returns a **generator object** — nothing executes until you iterate.

In [ ]:
def countdown(n: int) -> Generator[int, None, None]:
    """Generator that counts down from n."""
    print("  🚀 Starting countdown...")
    while n > 0:
        yield n    # ← Pause here, return n
        n -= 1     # ← Resume here on next iteration
    print("  💥 Liftoff!")

# Creating — NOTHING happens yet
gen = countdown(5)
print(f"Type: {type(gen)}")
print("Generator created, but no output yet!\n")

# Iterating — runs the code
for num in gen:
    print(f"    {num}")

## 5.2 `next()` — Manual Iteration

You can step through a generator manually with `next()`.

In [ ]:
def simple_gen():
    yield "first"
    yield "second"
    yield "third"

g = simple_gen()

print(next(g))   # first
print(next(g))   # second
print(next(g))   # third

try:
    next(g)        # StopIteration — exhausted!
except StopIteration:
    print("Generator exhausted!")

## 5.3 Memory Efficiency — Lists vs Generators

This is the **main reason** generators exist: processing large datasets with **O(1) memory**.

In [ ]:
def numbers_list(n: int) -> list[int]:
    """Returns all numbers — O(n) memory."""
    return [i for i in range(n)]

def numbers_gen(n: int) -> Generator[int, None, None]:
    """Yields numbers one by one — O(1) memory."""
    for i in range(n):
        yield i

# Compare memory usage
n = 100_000
list_size = sys.getsizeof(numbers_list(n))
gen_size = sys.getsizeof(numbers_gen(n))

print(f"List of {n:,} ints: {list_size:,} bytes ({list_size/1024:.1f} KB)")
print(f"Generator object:  {gen_size} bytes")
print(f"Ratio: list is {list_size/gen_size:.0f}x larger!")

## 5.4 Generator Expressions

Like list comprehensions, but with `()` instead of `[]` — lazy evaluation.

In [ ]:
# List comprehension — all values computed immediately
squares_list = [x**2 for x in range(10)]

# Generator expression — values computed on demand
squares_gen = (x**2 for x in range(10))

print(f"List: {squares_list}")
print(f"Generator: {squares_gen}")     # Shows object, not values
print(f"Values: {list(squares_gen)}")    # Consume into list

# Generator expressions in function calls
total = sum(x**2 for x in range(1000))   # No extra list in memory!
print(f"\nSum of squares 0-999: {total}")

# Chained with conditions
even_squares = (x**2 for x in range(20) if x % 2 == 0)
print(f"Even squares: {list(even_squares)}")

## 5.5 `yield from` — Delegating to Sub-generators

`yield from` delegates iteration to another iterable or generator.

In [ ]:
def gen_123():
    yield 1
    yield 2
    yield 3

def gen_abc():
    yield "a"
    yield "b"
    yield "c"

# Without yield from
def combined_manual():
    for item in gen_123():
        yield item
    for item in gen_abc():
        yield item

# With yield from — cleaner!
def combined():
    yield from gen_123()
    yield from gen_abc()

print(list(combined()))   # [1, 2, 3, 'a', 'b', 'c']

# Also works with any iterable
def flatten(nested: list) -> Generator:
    """Recursively flatten nested lists."""
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)   # Recursive delegation
        else:
            yield item

nested = [1, [2, 3], [4, [5, 6]], 7]
print(list(flatten(nested)))   # [1, 2, 3, 4, 5, 6, 7]

## 5.6 Infinite Generators

Generators can produce **infinite sequences** — only computed as needed.

In [ ]:
def naturals(start: int = 1) -> Generator[int, None, None]:
    """Infinite sequence of natural numbers."""
    n = start
    while True:
        yield n
        n += 1

# Take first 10 numbers
first_10 = list(islice(naturals(), 10))
print(f"First 10: {first_10}")

# Fibonacci — infinite
def fibonacci() -> Generator[int, None, None]:
    """Infinite Fibonacci sequence."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

first_15_fib = list(islice(fibonacci(), 15))
print(f"First 15 Fibonacci: {first_15_fib}")

## 5.7 `.send()` — Two-Way Communication

Generators aren't just output — you can **send values into** them.
The sent value becomes the result of the `yield` expression.

In [ ]:
def running_average():
    """Generator that computes a running average.
    Send values in, get current average out."""
    total = 0
    count = 0
    average = None
    
    while True:
        value = yield average  # Receive value, send back average
        if value is not None:
            total += value
            count += 1
            average = total / count

# Create and prime the generator
avg = running_average()
next(avg)  # Prime it — must call next() first to reach the yield

print(f"Send 10: avg = {avg.send(10)}")
print(f"Send 20: avg = {avg.send(20)}")
print(f"Send 30: avg = {avg.send(30)}")
print(f"Send 15: avg = {avg.send(15)}")

In [ ]:
# Practical: state machine with send()
def traffic_light():
    """State machine — send 'next' to change light."""
    states = ["🔴 RED", "🟢 GREEN", "🟡 YELLOW"]
    index = 0
    while True:
        command = yield states[index]
        if command == "next":
            index = (index + 1) % len(states)

light = traffic_light()
print(next(light))           # 🔴 RED (initial state)
print(light.send("next"))    # 🟢 GREEN
print(light.send("next"))    # 🟡 YELLOW
print(light.send("next"))    # 🔴 RED
print(light.send("stay"))    # 🔴 RED (no change)

print("\n💡 .send() is how @contextmanager and async/await work under the hood")

## 5.8 Practical: Pipeline Pattern

Chain generators for efficient data processing — like Unix pipes.

In [ ]:
# Data pipeline with generators — each step is lazy

def read_data():
    """Step 1: Simulate reading lines from a file."""
    data = ["  Alice, 30  ", "  Bob, 25  ", "  ", "  Charlie, 35  ", ""]
    yield from data

def clean(lines):
    """Step 2: Strip whitespace, skip empty lines."""
    for line in lines:
        cleaned = line.strip()
        if cleaned:
            yield cleaned

def parse(lines):
    """Step 3: Parse 'name, age' into dicts."""
    for line in lines:
        name, age = line.split(", ")
        yield {"name": name, "age": int(age)}

def adults_only(records):
    """Step 4: Filter to age >= 30."""
    for record in records:
        if record["age"] >= 30:
            yield record

# Build the pipeline — nothing runs yet!
pipeline = adults_only(parse(clean(read_data())))

# Consume — runs all stages lazily
for record in pipeline:
    print(f"  {record}")

## 5.9 Essential `itertools`

The `itertools` module provides generator-based building blocks.
These are the most commonly used ones:

| Function | Purpose | Java equivalent |
|----------|---------|----------------|
| `islice(gen, n)` | Take first N items | `stream.limit(n)` |
| `chain(a, b)` | Concatenate iterables | `Stream.concat()` |
| `takewhile(pred, it)` | Take while condition is true | `stream.takeWhile()` |
| `count(start)` | Infinite counter | `Stream.iterate()` |
| `groupby(it, key)` | Group consecutive items | `Collectors.groupingBy()` |

In [ ]:
# chain — merge multiple iterables into one
letters = chain("abc", "def", "ghi")
print(f"chain:     {list(letters)}")

# count — infinite counter (like naturals() but built-in)
evens = (x for x in count(0) if x % 2 == 0)
print(f"count:     {list(islice(evens, 8))}")

# takewhile — take items while condition holds
nums = [1, 3, 5, 2, 4, 6, 8]
odd_prefix = list(takewhile(lambda x: x % 2 == 1, nums))
print(f"takewhile: {odd_prefix}")  # Stops at first even (2)

# groupby — group consecutive equal items
data = "AAABBBCCAAB"
groups = [(key, list(group)) for key, group in groupby(data)]
print(f"groupby:   {groups}")

In [ ]:
# Real-world: batch processing with islice
def batch(iterable, size: int):
    """Split any iterable into chunks of `size`."""
    it = iter(iterable)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

items = range(17)
for i, chunk in enumerate(batch(items, 5)):
    print(f"  Batch {i+1}: {chunk}")

print("\n💡 This pattern is used for: DB bulk inserts, API pagination, file chunking")

## ⚠️ Generators Are Single-Use!

Once exhausted, a generator cannot be restarted. You must create a new one.

In [ ]:
gen = (x**2 for x in range(5))

print(f"First pass:  {list(gen)}")    # [0, 1, 4, 9, 16]
print(f"Second pass: {list(gen)}")    # [] — exhausted!

## 5.10 Exercises

**Exercise 1:** Write a `chunked_reader(text, chunk_size)` generator that yields chunks of text.
E.g., `chunked_reader("HelloWorld", 3)` → `"Hel"`, `"loW"`, `"orl"`, `"d"`

In [ ]:
# Your solution here


In [ ]:
# ✅ Solution
def chunked_reader(text: str, chunk_size: int) -> Generator[str, None, None]:
    """Yield text in chunks of chunk_size."""
    for i in range(0, len(text), chunk_size):
        yield text[i:i + chunk_size]

for chunk in chunked_reader("HelloWorld", 3):
    print(f"  '{chunk}'")

# Also works with large strings — O(1) memory
big_text = "A" * 1_000_000
chunks = chunked_reader(big_text, 10_000)
print(f"\nChunks: {sum(1 for _ in chunks)} × 10,000 chars")

**Exercise 2:** Write an infinite `unique_id(prefix)` generator that yields IDs like `"user_001"`, `"user_002"`, etc. Use it with `islice` to get 5 IDs.

In [ ]:
# Your solution here


In [ ]:
# ✅ Solution
def unique_id(prefix: str = "id") -> Generator[str, None, None]:
    """Infinite generator of unique IDs."""
    n = 1
    while True:
        yield f"{prefix}_{n:03d}"
        n += 1

user_ids = list(islice(unique_id("user"), 5))
print(f"Users:    {user_ids}")

order_ids = list(islice(unique_id("order"), 5))
print(f"Orders:   {order_ids}")

# Each generator has independent state
gen1 = unique_id("a")
gen2 = unique_id("b")
print(f"\ngen1: {next(gen1)}, gen2: {next(gen2)}, gen1: {next(gen1)}")

## 📋 Takeaways

| # | Concept | Key Point |
|---|---------|----------|
| 1 | `yield` | Pauses function, returns value, resumes on next call |
| 2 | Lazy evaluation | Nothing runs until you iterate |
| 3 | O(1) memory | Only one value in memory at a time |
| 4 | Generator expression | `(x**2 for x in range(n))` — lazy list comp |
| 5 | `yield from` | Delegate to sub-generator or iterable |
| 6 | `next(gen)` | Manual iteration, raises `StopIteration` when done |
| 7 | `.send(value)` | Two-way: send values *into* generators |
| 8 | Single-use | Cannot restart — create a new generator |
| 9 | Pipeline pattern | Chain generators for memory-efficient processing |
| 10 | `itertools` | `islice`, `chain`, `takewhile`, `count`, `groupby` |
| 11 | `batch()` pattern | `islice` in a loop for chunked processing |
| 12 | Java comparison | `yield` ≈ `Iterator` — but without the class boilerplate |